In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from functools import reduce

In [2]:
class MLP(nn.Module):
    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS):
        super().__init__()
        self.activation = nn.Tanh()
        self.mpl_linear_connect = self._initialize_network(N_INPUT, N_HIDDEN, N_OUTPUT, N_LAYERS)

    def _initialize_network(self,N_INPUT, N_HIDDEN, N_OUTPUT, N_LAYERS):

        layer_list = (
            [nn.Linear( N_INPUT, N_HIDDEN)] + 
            [nn.Linear( N_HIDDEN, N_HIDDEN) for _ in range(N_LAYERS-2)] + 
            [nn.Linear(N_HIDDEN, N_OUTPUT)]
        )

        for layer in layer_list:
            nn.init.xavier_uniform_(layer.weight, gain=nn.init.calculate_gain("tanh"))
            nn.init.constant_(layer.bias, 0)
        return nn.ModuleList(layer_list)

    def forward(self, X):
        for layer in self.mpl_linear_connect[:-1]:  
            X = self.activation( layer(X) )
        X = self.mpl_linear_connect[-1](X)

        return X

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

# =========================
# Parameters
# =========================
alpha, beta, gamma, delta = 3., 1., 1.5, 1.
x0, y0 = 1., 1.

# =========================
# Feature transform (KEY FIX)
# =========================
def feature_transform(t):
    return torch.cat([
        torch.sin(t),
        torch.sin(2*t),
        torch.sin(3*t),
        torch.sin(4*t),
    ], dim=1)

# =========================
# Output transform (hard IC)
# =========================
def output_transform(t, y):
    y1 = y[:, 0:1]
    y2 = y[:, 1:2]

    x = y1 * torch.tanh(t) + x0
    y = y2 * torch.tanh(t) + y0

    return x, y

# =========================
# MLP (Tanh works better here)
# =========================
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 2)
        )

    def forward(self, t):
        t_feat = feature_transform(t)
        raw = self.net(t_feat)
        return output_transform(t, raw)

model = MLP()

# =========================
# Training points
# =========================
t = torch.linspace(0, 5, 200).view(-1,1)
t.requires_grad_(True)

# =========================
# Training
# =========================
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5000):
    opt.zero_grad()

    x, y = model(t)

    dx = torch.autograd.grad(x, t, torch.ones_like(x), create_graph=True)[0]
    dy = torch.autograd.grad(y, t, torch.ones_like(y), create_graph=True)[0]

    f1 = dx - (alpha*x - beta*x*y)
    f2 = dy - (delta*x*y - gamma*y)

    loss = (f1**2).mean() + (f2**2).mean()

    loss.backward()
    opt.step()

    if epoch % 500 == 0:
        print(epoch, loss.item())

# =========================
# LBFGS (CRITICAL)
# =========================
opt_lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=500)

def closure():
    opt_lbfgs.zero_grad()

    x, y = model(t)

    dx = torch.autograd.grad(x, t, torch.ones_like(x), create_graph=True)[0]
    dy = torch.autograd.grad(y, t, torch.ones_like(y), create_graph=True)[0]

    f1 = dx - (alpha*x - beta*x*y)
    f2 = dy - (delta*x*y - gamma*y)

    loss = (f1**2).mean() + (f2**2).mean()
    loss.backward()
    return loss

opt_lbfgs.step(closure)

# =========================
# Reference solution
# =========================
def lv(t, z):
    x, y = z
    return [alpha*x - beta*x*y, delta*x*y - gamma*y]

t_eval = np.linspace(0,5,200)
sol = solve_ivp(lv, [0,5], [x0,y0], t_eval=t_eval)

# =========================
# Prediction
# =========================
with torch.no_grad():
    t_test = torch.tensor(t_eval, dtype=torch.float32).view(-1,1)
    x_pred, y_pred = model(t_test)

    x_pred = x_pred.numpy()
    y_pred = y_pred.numpy()

# =========================
# Plot
# =========================
plt.figure()
plt.plot(sol.t, sol.y[0], label="x solve_ivp")
plt.plot(sol.t, sol.y[1], label="y solve_ivp")
plt.plot(sol.t, x_pred, '--', label="x PINN")
plt.plot(sol.t, y_pred, '--', label="y PINN")
plt.legend()
plt.show()

In [ ]:
"""Backend supported: tensorflow.compat.v1, tensorflow, pytorch, paddle, jax"""
import deepxde as dde
import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate

import torch


ub = 200
rb = 20


def func(t, r):
    x, y = r
    dx_t = 1 / ub * rb * (2.0 * ub * x - 0.04 * ub * x * ub * y)
    dy_t = 1 / ub * rb * (0.02 * ub * x * ub * y - 1.06 * ub * y)
    return dx_t, dy_t


def gen_truedata():
    t = np.linspace(0, 1, 100)

    sol = integrate.solve_ivp(func, (0, 10), (100 / ub, 15 / ub), t_eval=t)
    x_true, y_true = sol.y
    x_true = x_true.reshape(100, 1)
    y_true = y_true.reshape(100, 1)

    return x_true, y_true


def ode_system(x, y):
    # Most backends
    r = y[:, 0:1]
    p = y[:, 1:2]
    dr_t = dde.grad.jacobian(y, x, i=0)
    dp_t = dde.grad.jacobian(y, x, i=1)

    return [
        dr_t - 1 / ub * rb * (2.0 * ub * r - 0.04 * ub * r * ub * p),
        dp_t - 1 / ub * rb * (0.02 * r * ub * p * ub - 1.06 * p * ub),
    ]

geom = dde.geometry.TimeDomain(0, 1.0)
data = dde.data.PDE(geom, ode_system, [], 3000, 2, num_test=3000)

layer_size = [1] + [64] * 6 + [2]
activation = "tanh"
initializer = "Glorot normal"
net = dde.nn.FNN(layer_size, activation, initializer)

# Backend pytorch
def input_transform(t):
     return torch.cat(
         [
             torch.sin(t),
         ],
         dim=1,
     )

# Backend pytorch
def output_transform(t, y):
     y1 = y[:, 0:1]
     y2 = y[:, 1:2]
     return torch.cat([y1 * torch.tanh(t) + 100 / ub, y2 * torch.tanh(t) + 15 / ub], dim=1)

net.apply_feature_transform(input_transform)
net.apply_output_transform(output_transform)
model = dde.Model(data, net)

model.compile("adam", lr=0.001)
losshistory, train_state = model.train(iterations=50000)
# Most backends except jax can have a second fine tuning of the solution
model.compile("L-BFGS")
losshistory, train_state = model.train()
dde.saveplot(losshistory, train_state, issave=True, isplot=True)

plt.xlabel("t")
plt.ylabel("population")

t = np.linspace(0, 1, 100)
x_true, y_true = gen_truedata()
plt.plot(t, x_true, color="black", label="x_true")
plt.plot(t, y_true, color="blue", label="y_true")

t = t.reshape(100, 1)
sol_pred = model.predict(t)
x_pred = sol_pred[:, 0:1]
y_pred = sol_pred[:, 1:2]

plt.plot(t, x_pred, color="red", linestyle="dashed", label="x_pred")
plt.plot(t, y_pred, color="orange", linestyle="dashed", label="y_pred")
plt.legend()
plt.show()